# 🤖 LLM Fundamentals for Business Managers
### Essential Knowledge Every Leader Should Have (15-20 minutes)

---
## ⚙️ Setup - Run This First

In [ ]:
!pip install litellm tiktoken python-dotenv ipython -q

import os
from dotenv import load_dotenv
from litellm import completion
import tiktoken
from IPython.display import Markdown, display, HTML

load_dotenv()

# Styling
NAVY = "#0B1F3A"
ICE = "#EAF2FF"
GOLD = "#B08D57"

def banner(text, color=NAVY):
    display(HTML(f"""
    <div style="background:{color};color:{ICE};padding:14px 22px;border-radius:8px;
                font-family:'Segoe UI',sans-serif;font-size:19px;font-weight:600;
                margin:14px 0 6px 0;letter-spacing:0.2px;">
        {text}
    </div>
    """))

def insight(text):
    display(HTML(f"""
    <div style="background:{ICE};border-left:5px solid {GOLD};padding:14px 20px;
                border-radius:4px;font-family:'Segoe UI',sans-serif;font-size:15.5px;
                color:{NAVY};margin:10px 0 22px 0;">
        <b>💡 Business Insight:</b> {text}
    </div>
    """))

def warning_box(text):
    display(HTML(f"""
    <div style="background:#FBEAEA;border-left:5px solid #B23B3B;padding:14px 20px;
                border-radius:4px;font-family:'Segoe UI',sans-serif;font-size:15.5px;
                color:#5A1A1A;margin:10px 0 22px 0;">
        <b>⚠️ Critical Warning:</b> {text}
    </div>
    """))

print("✅ Setup complete. You are ready to go!")

---
## 1. What Are LLMs Really Doing?

Large Language Models are **advanced next-word predictors**. 
They don’t “understand” — they predict the most likely next token based on patterns learned during training.

---
## 2. Tokens — The Real Unit of Cost

In [ ]:
enc = tiktoken.encoding_for_model("gpt-4o")

examples = {
    "One-line status update": "Backup job 4471 completed successfully at 2:03 AM.",
    "Short incident note": "Nightly backup for East Region cluster failed due to network timeout.",
    "Full postmortem excerpt": "At 01:47 AM, the scheduled backup for the East Region storage cluster failed..."
}

banner("Token Cost by Document Type")

for label, text in examples.items():
    words = len(text.split())
    tokens = len(enc.encode(text))
    print(f"{label:<35} {words:>3} words → {tokens:>3} tokens")

insight("You are billed **per token**, not per word. A 10-page document (~5,000 tokens) summarized with gpt-4o-mini costs less than a penny.")

---
## 3. Temperature: Predictable vs Creative

In [ ]:
prompt = "Draft a one-paragraph internal summary for leadership about a recurring backup failure on the East Region cluster."

for temp in [0.0, 0.7, 1.4]:
    banner(f"Temperature = {temp} {'(Strict & Consistent)' if temp == 0.0 else '(Balanced)' if temp == 0.7 else '(Creative & Exploratory)'}", color=GOLD if temp > 0.5 else NAVY)
    resp = completion(model="gpt-4o", messages=[{"role": "user", "content": prompt}], temperature=temp)
    display(Markdown(resp.choices[0].message.content.strip()))
    print("─" * 80)

insight("**Low temperature (0.0-0.3)** → Best for finance, legal, compliance, reports.<br>**Higher temperature (0.7-1.0)** → Best for brainstorming, marketing, creative content.")

---
## 4. ⚠️ The Hallucination Warning

In [ ]:
warning_box("LLMs can generate fluent, confident, but completely false information.<br><br>"
            "<b>Real business risks:</b> Made-up policies, fake court cases, incorrect numbers, non-existent features.<br><br>"
            "<b>Rule:</b> Always verify critical outputs with human oversight.")

---
## 5. System Prompts — Turn One Assistant into Many Experts

In [ ]:
def ask(system_prompt, user_prompt, model="gpt-4o-mini", temperature=0.0):
    resp = completion(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        temperature=temperature
    )
    return resp.choices[0].message.content.strip()

ticket = "Customer's nightly backup failed three times due to storage quota limit. Resolved by increasing capacity."

banner("Internal Technical Summarizer")
internal = "You are a concise IT support analyst. Be technical, factual, and structured."
display(Markdown(ask(internal, f"Summarize: {ticket}")))

banner("Customer-Facing Communicator", color=GOLD)
customer = "You are a warm, reassuring customer success manager. Avoid blame and use simple language."
display(Markdown(ask(customer, f"Write a short email to the customer about: {ticket}")))

insight("A good **system prompt** is like giving a new hire a clear job description. It dramatically changes tone, depth, and focus.")

---
## 6. Same Question, Different Models + Real Cost Tracking

In [ ]:
total_cost = 0.0

def compare_models(prompt):
    global total_cost
    models = ["gpt-4o", "anthropic/claude-haiku-4-5"]
    
    for model in models:
        try:
            resp = completion(model=model, messages=[{"role": "user", "content": prompt}], temperature=0.0)
            
            prompt_tokens = getattr(resp.usage, 'prompt_tokens', 0)
            completion_tokens = getattr(resp.usage, 'completion_tokens', 0)
            cost = litellm.completion_cost(resp)
            total_cost += cost
            
            banner(f"Model: {model}", color=GOLD)
            print(f"Input tokens : {prompt_tokens}")
            print(f"Output tokens: {completion_tokens}")
            print(f"This call    : ${cost:.6f}")
            print(f"Running total: ${total_cost:.6f}")
            display(Markdown(resp.choices[0].message.content.strip()))
            print("─" * 80)
            
        except Exception as e:
            print(f"Error with {model}: {e}")

compare_models("Summarize the key benefits and risks of using generative AI in our company.")

---
## Final Takeaways for Managers

1. **LLMs predict tokens**, they don’t retrieve facts.
2. **Tokens = money** — longer documents cost more.
3. Use **temperature** deliberately: low for precision, higher for creativity.
4. **Hallucinations are inevitable** — always verify critical content.
5. **System prompts** are your most powerful control tool.
6. Different models = different strengths and costs.
7. Combine **great prompting + human oversight** for best results.

**You don’t need to become a prompt engineer — you just need to know what to watch for.**